# 04 - Lien clustering / classification

In [1]:
import logging
import os
import warnings

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from xgboost import XGBClassifier

logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="mlflow")
sns.set_theme(style="whitegrid")

## Configuration et données

In [2]:
DATA_PATH = "data/processed/clean_eng.csv"
FIG_DIR = "reports/figures"
SEED = 42
MLFLOW_URI = "sqlite:///mlflow.db"
EXPERIMENT = "churn-prediction"
SCORING = ["accuracy", "precision", "recall", "f1", "roc_auc"]
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

NUMERIC = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
ENGINEERED = ["charges_per_tenure", "is_new_customer", "is_month_to_month", "is_fiber", "is_electronic_check", "nb_services"]
ORDINAL = {
    "Contract": ["Month-to-month", "One year", "Two year"],
    "MultipleLines": ["No phone service", "No", "Yes"],
    "InternetService": ["No", "DSL", "Fiber optic"],
}
NOMINAL = [
    "gender", "Partner", "Dependents", "PhoneService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "PaperlessBilling", "PaymentMethod",
]

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)

df = pd.read_csv(DATA_PATH)
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["Churn", "customerID"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("train:", X_train.shape, "test:", X_test.shape)
print("churn train / test:", round(y_train.mean(), 3), "/", round(y_test.mean(), 3))

train: (5634, 26) test: (1409, 26)
churn train / test: 0.265 / 0.265


### Preprocessing

In [3]:
def build_preprocessor(use_fe=True, use_cluster=False):
    num_cols = NUMERIC + (ENGINEERED if use_fe else [])
    nom_cols = NOMINAL + (["cluster"] if use_cluster else [])
    numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    ordinal = make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories=list(ORDINAL.values()), handle_unknown="use_encoded_value", unknown_value=-1),
    )
    nominal = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", drop="if_binary"))
    return ColumnTransformer(
        [("num", numeric, num_cols), ("ord", ordinal, list(ORDINAL)), ("nom", nominal, nom_cols)],
        verbose_feature_names_out=False,
    )

### Variable `cluster`

In [4]:
N_CLUSTERS = 3
CLUSTER_NUMERIC = ["SeniorCitizen", "tenure", "MonthlyCharges"] + ENGINEERED
CLUSTER_CATEGORICAL = list(ORDINAL) + NOMINAL + ["tenure_band"]


def build_cluster_pipeline(k=N_CLUSTERS):
    prep = ColumnTransformer(
        [
            ("num", StandardScaler(), CLUSTER_NUMERIC),
            ("cat", make_pipeline(OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), StandardScaler()), CLUSTER_CATEGORICAL),
        ]
    )
    return make_pipeline(prep, KMeans(n_clusters=k, n_init=10, random_state=SEED))


class ClusterFeature(BaseEstimator, TransformerMixin):
    def __init__(self, k=N_CLUSTERS):
        self.k = k

    def fit(self, X, y=None):
        self.model_ = build_cluster_pipeline(self.k).fit(X)
        return self

    def transform(self, X):
        out = X.copy()
        out["cluster"] = self.model_.predict(X)
        return out